# An End-to-End AI Harvest Planner for Low-Cost Fruit-Picking Robots Built on a Physics-Consistent World Model

**`03_evaluation.ipynb`**

What the finished system does, and what each part of it is worth.

Three kinds of number, kept apart because they answer different questions.

**Component.** How well the outcome model predicts a pick it has not seen. This is the only part
with labels, so it is the only part with an accuracy.

**System.** What the assembled chain harvests from a tree, against the alternatives it could have
used instead. The planner and the pick policy have no labels — they choose actions — so they are
measured by the task, not by prediction error.

**Fidelity.** Whether those probabilities hold up when the physics gets a say. That is
`04_fidelity`; the calibration here is the part that can be checked without re-running MuJoCo.

**Everything is measured at the shipping configuration**: twenty stops from the trained planner,
a sweep stage for whatever those twenty miss, no selection threshold. An earlier version of this
notebook reported a comparison at three stops, which was the operating point at the time; three
stops reach a third of what the arm can touch and is now a different machine, so those figures
are not carried forward.

**Outcomes are sampled here, not taken as expectations.** `06_orchard_estimate` reports what the
plan intends; this reports what a draw from the outcome model gives, with a knocked neighbour
actually removed so the rest of the tree is planned around the canopy that is left. Thirty seeds,
and comparisons paired on the seed so that seed-to-seed variation cancels rather than hiding a
difference of one or two fruit.


In [1]:
import os, sys, time, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Default to the folder this notebook sits in, so a clone runs without setup. An absolute
# default only ever pointed at one machine, and an environment variable set in a shell does
# not reach a kernel that was already running.
ROOT = Path(os.environ.get("AIPICK_ROOT") or Path.cwd())
os.environ["AIPICK_ROOT"] = str(ROOT)
SRC, DATA, MODELS = ROOT/"src", ROOT/"data", ROOT/"models"
OUT = ROOT/"runs"/"evaluation"; OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(SRC))

import environment as E
E.load(ROOT, trees="trees_measured_pose.csv", dynamics=True)

import planner as PL
info = PL.load(ROOT)

TREES_TRAIN, TREES_EVAL = list(range(0, 20)), list(range(20, 50))
N_SEEDS = 30

plt.rcParams.update({"figure.dpi": 130, "savefig.dpi": 200, "savefig.bbox": "tight",
                     "font.size": 9, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.width", 220)

print(f"canopy {len(E.T):,} fruit / {E.T.tree.nunique()} trees")
print(f"planner {info['planner']}   policy {info['policy']}   scaling {info['scaling']}")
print(f"configuration: {PL.K_STATIONS} stops + sweep, threshold {PL.THRESHOLD}, "
      f"arm {PL.HALF_X} x {PL.LIFT} m")
print(f"held-out trees {TREES_EVAL[0]}-{TREES_EVAL[-1]}, {N_SEEDS} seeds")



c:\python\Lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


canopy 6,000 fruit / 50 trees
planner station_planner_k20.pt   policy pick_policy.pt   scaling recomputed
configuration: 20 stops + sweep, threshold 0.0, arm 0.2 x 0.6 m
held-out trees 20-49, 30 seeds


## 1. The outcome model on picks it has not seen

The split comes from the detection manifest, so it is by fruit: all four aperture rows for a
fruit fall on the same side. Splitting by row would put three of a fruit's four apertures in
training and score the fourth as though it were new.

Picks where the solver diverged are dropped — a palm that ended two metres from its seat, a
thirty-second pick, four fingers on a three-fingered gripper. Those are not hard cases, they are
numbers with no physical meaning, and they are dropped by fruit so the counterfactual structure
survives.


In [2]:
import pickle

D = pd.read_csv(DATA/"picks.csv")
MAN = pd.read_csv(ROOT/"apple_crops"/"manifest.csv")
D["apple_id"] = D.apple_id.astype(str)
MAN["apple_id"] = MAN.apple_id.astype(str)
if "split" not in D:
    D = D.merge(MAN[["apple_id", "split"]], on="apple_id", how="left")

# Four of the model's inputs are derived rather than stored: three are the raw observation with
# its sentinel replaced by the fill the training pipeline used, and one is a flag. Rebuilding
# them here rather than saving them keeps picks.csv as the record of what the simulator
# produced, but it does mean this recipe has to match the one in 01 exactly.
for c, base in (("obs_nearest_diam_c", "obs_nearest_diam"),
                ("obs_nb_along_c", "obs_nb_along"),
                ("obs_nb_radial_c", "obs_nb_radial")):
    if c not in D.columns and base in D.columns:
        v = D[base].to_numpy(float)
        D[c] = np.where(v == E.SENTINEL, E.FILLS[base], v)
if "obs_has_neighbour" not in D.columns and "obs_n_neighbours" in D.columns:
    D["obs_has_neighbour"] = (D.obs_n_neighbours.to_numpy(float) > 0).astype(float)

DAMAGE = ("STEM_PULL", "STALK_SNAP", "SPUR_BREAK")
D["y"] = D.code.map(lambda c: "DEFECT" if c in DAMAGE else c)

bad = set(D.loc[(D.seat_x_palm > 2.0) | D.seat_x_palm.isna() |
                (D.secs > 30) | (D.n_fingers > 3), "apple_id"])
D = D[~D.apple_id.isin(bad)]
missing = [c for c in E.FEATS if c not in D.columns]
assert not missing, f"cannot rebuild {missing}"
print(f"{len(D):,} rows over {D.apple_id.nunique():,} fruit "
      f"({len(bad):,} fruit dropped as divergent)\n")

TE = D[D.split == "test"].copy()
proba = E.MODEL.predict_proba(TE[E.FEATS].to_numpy(float))
TE["pred"] = np.array(E.CLASSES)[proba.argmax(1)]
acc = (TE.pred == TE.y).mean()*100
print(f"held-out accuracy {acc:.2f}%   ({len(TE):,} rows / {TE.apple_id.nunique():,} fruit)")

TE["p_top"] = proba.max(1)
TE["hit"] = (TE.pred == TE.y).astype(float)
dec = TE.assign(band=pd.qcut(TE.p_top, 10, duplicates="drop")).groupby(
    "band", observed=True).agg(n=("hit", "size"), confidence=("p_top", "mean"),
                               accuracy=("hit", "mean"))
dec["gap"] = (dec.confidence - dec.accuracy).round(3)
print(f"\ncalibration by decile of top-class confidence\n")
print(dec.round(3).to_string())
print(f"\n  worst gap {dec.gap.abs().max():.3f}")


31,960 rows over 7,990 fruit (143 fruit dropped as divergent)

held-out accuracy 94.07%   (7,052 rows / 1,763 fruit)

calibration by decile of top-class confidence

                    n  confidence  accuracy    gap
band                                              
(0.265, 0.8554]   706       0.670     0.644  0.026
(0.8554, 0.9457]  705       0.911     0.887  0.025
(0.9457, 0.972]   705       0.961     0.942  0.019
(0.972, 0.9835]   705       0.978     0.974  0.004
(0.9835, 0.9898]  705       0.987     0.990 -0.003
(0.9898, 0.9931]  705       0.992     0.987  0.004
(0.9931, 0.9947]  705       0.994     0.991  0.002
(0.9947, 0.9955]  705       0.995     0.999 -0.003
(0.9955, 0.9968]  705       0.996     0.997 -0.001
(0.9968, 0.997]   706       0.997     0.996  0.001

  worst gap 0.026


## 2. The planner against the search it replaces

Both choose twenty stops and both hand what they miss to the sweep stage. The question is not
which reaches more fruit — with the sweep on, both reach everything the arm can touch — but how
much of that is the network's doing and how much the sweep is covering for.

`06_orchard_estimate` runs this on expectations. Here the outcomes are sampled, which is what
puts an interval on the difference.


In [3]:
t0 = time.time()
RUNS = {}
for tag, kw in (("planner", dict(chooser="planner")),
                ("coverage", dict(chooser="greedy"))):
    RUNS[tag] = PL.realise_summary(TREES_EVAL, seeds=range(N_SEEDS), **kw)
    print(f"  {tag:<10} {len(RUNS[tag]):,} tree-seeds   {(time.time()-t0)/60:5.1f} min")

SYS = pd.concat([df.assign(config=t) for t, df in RUNS.items()], ignore_index=True)
SYS.to_csv(OUT/"per_tree_seed.csv", index=False)

G = SYS.groupby("config", sort=False).agg(
    stops=("stops", "mean"), stage1=("stage1_stops", "mean"), sweep=("sweep_stops", "mean"),
    attempts=("attempts", "mean"), premium=("premium", "mean"),
    knocked=("knocked", "mean"), utility=("utility", "mean"),
    seconds=("seconds", "mean")).round(2)
print(f"\nper tree, sampled outcomes, {N_SEEDS} seeds x {len(TREES_EVAL)} trees\n")
print(G.to_string())


  planner    900 tree-seeds     3.6 min
  coverage   900 tree-seeds     6.4 min

per tree, sampled outcomes, 30 seeds x 30 trees

          stops  stage1  sweep  attempts  premium  knocked  utility  seconds
config                                                                      
planner   21.27    20.0   1.27     62.07    49.61     3.28    45.57  1218.59
coverage  21.33    18.3   3.03     62.07    49.61     3.28    45.57  1219.38


In [4]:
def paired(a, b):
    '''Paired on the tree-seed pair, which is the unit both configurations share.'''
    a, b = np.asarray(a, float), np.asarray(b, float)
    d = a - b
    n, mean, sd = len(d), d.mean(), d.std(ddof=1)
    try:
        from scipy import stats as ss
        _, p = ss.ttest_rel(a, b)
        half = ss.t.ppf(0.975, n-1)*sd/np.sqrt(n)
        lo, hi = mean-half, mean+half
    except Exception:
        rg = np.random.default_rng(0)
        bs = rg.choice(d, (20000, n), replace=True).mean(axis=1)
        lo, hi = np.percentile(bs, [2.5, 97.5])
        p = 2*min((bs <= 0).mean(), (bs >= 0).mean())
    return mean, lo, hi, p, (mean/sd if sd else np.nan)


key = ["tree", "seed"]
A = RUNS["planner"].set_index(key).sort_index()
B = RUNS["coverage"].set_index(key).sort_index()
assert A.index.equals(B.index)

print(f"planner minus coverage search, paired on tree and seed ({len(A):,} pairs)\n")
print(f"  {'':<14}{'difference':>12}{'95% CI':>22}{'p':>9}{'d':>7}")
for col, label in (("premium", "premium"), ("utility", "utility"),
                   ("attempts", "attempts"), ("seconds", "seconds"),
                   ("sweep_stops", "sweep stops")):
    m, lo, hi, p, d = paired(A[col], B[col])
    print(f"  {label:<14}{m:>+12.3f}{f'[{lo:+.3f}, {hi:+.3f}]':>22}{p:>9.4f}{d:>7.2f}")

print("\n  The two finish in the same place, and the table says so: the premium difference is")
print("  a fraction of a fruit. What separates them is the sweep -- the planner spends all")
print("  twenty of its stops and hands the sweep less to do, while coverage search stops at")
print("  its first zero-gain step. The network's contribution here is that it needs no")
print("  coverage matrix at run time, not that it places stops better.")


planner minus coverage search, paired on tree and seed (900 pairs)

                  difference                95% CI        p      d
  premium             +0.000      [+0.000, +0.000]      nan    nan
  utility             +0.000      [+0.000, +0.000]      nan    nan
  attempts            +0.000      [+0.000, +0.000]      nan    nan
  seconds             -0.787      [-1.413, -0.162]   0.0137  -0.08
  sweep stops         -1.767      [-1.869, -1.664]   0.0000  -1.13

  The two finish in the same place, and the table says so: the premium difference is
  a fraction of a fruit. What separates them is the sweep -- the planner spends all
  twenty of its stops and hands the sweep less to do, while coverage search stops at
  its first zero-gain step. The network's contribution here is that it needs no
  coverage matrix at run time, not that it places stops better.


## 3. Three ways to choose the next fruit

The shipping configuration uses the rate rule. The pick policy was trained three times over — at
three stops, at twenty, and with the clock removed and damage charged — and this is where the
three operating points sit next to each other rather than being described one at a time.

None of these is a failure. They optimise different things, and a grower with a crew already in
the block would not pick the same one as a grower who cannot find anybody.


In [5]:
t0 = time.time()
PICKERS = {}
for tag, kw in (("rate rule", dict(picker="rule")),
                ("pick policy", dict(picker="policy")),
                ("threshold 0.9", dict(picker="rule", threshold=0.9))):
    PICKERS[tag] = PL.realise_summary(TREES_EVAL, seeds=range(N_SEEDS), **kw)
    print(f"  {tag:<15} {(time.time()-t0)/60:5.1f} min")

P = pd.concat([df.assign(picker=t) for t, df in PICKERS.items()], ignore_index=True)
P.to_csv(OUT/"pickers.csv", index=False)

T = P.groupby("picker", sort=False).agg(
    attempts=("attempts", "mean"), premium=("premium", "mean"),
    knocked=("knocked", "mean"), utility=("utility", "mean"),
    seconds=("seconds", "mean")).round(2)
T["minutes"] = (T.seconds/60).round(1)
T["premium_per_hour"] = (T.premium/(T.seconds + E.TREE_SPACING/E.TRAVEL)*3600).round(0)
T["s_per_premium"] = (T.seconds/T.premium.clip(lower=0.01)).round(1)
print(f"\nper tree, {N_SEEDS} seeds\n")
print(T[["attempts", "premium", "knocked", "utility", "minutes",
         "premium_per_hour", "s_per_premium"]].to_string())

base = PICKERS["rate rule"].set_index(key).sort_index()
print(f"\n\nagainst the rate rule, paired\n")
print(f"  {'':<16}{'premium':>11}{'knocked':>10}{'utility':>10}{'minutes':>10}")
for tag in ("pick policy", "threshold 0.9"):
    o = PICKERS[tag].set_index(key).sort_index()
    print(f"  {tag:<16}{o.premium.mean()-base.premium.mean():>+11.2f}"
          f"{o.knocked.mean()-base.knocked.mean():>+10.2f}"
          f"{o.utility.mean()-base.utility.mean():>+10.2f}"
          f"{(o.seconds.mean()-base.seconds.mean())/60:>+10.1f}")
print("\n  The rule takes the most fruit, which is what this client asked for. The policy takes")
print("  about two thirds of it in half the time and knocks a twentieth as many neighbours;")
print("  the threshold sits between them. Utility -- the grade-weighted total -- is what")
print("  decides, and on that the rule wins because premium fruit dominate the table.")


  rate rule         5.9 min
  pick policy      15.4 min
  threshold 0.9    19.6 min

per tree, 30 seeds

               attempts  premium  knocked  utility  minutes  premium_per_hour  s_per_premium
picker                                                                                      
rate rule         62.07    49.61     3.28    45.57     20.3             146.0           24.6
pick policy       34.31    33.35     0.12    33.17      8.7             227.0           15.7
threshold 0.9     37.88    37.03     0.09    36.88     12.2             181.0           19.8


against the rate rule, paired

                      premium   knocked   utility   minutes
  pick policy          -16.25     -3.16    -12.40     -11.6
  threshold 0.9        -12.57     -3.19     -8.70      -8.1

  The rule takes the most fruit, which is what this client asked for. The policy takes
  about two thirds of it in half the time and knocks a twentieth as many neighbours;
  the threshold sits between them. Utility -

## 4. Does it hold on trees it was not fitted to?

The station planner was trained on trees 0 to 19 and everything above is measured on 20 to 49.
Running both sets through the same configuration says whether the gap between them is a
generalisation gap or just the trees being different.


In [6]:
t0 = time.time()
GEN = {}
for tag, tids in (("training trees", TREES_TRAIN), ("held-out trees", TREES_EVAL)):
    GEN[tag] = PL.realise_summary(tids, seeds=range(10))
    print(f"  {tag:<16} {len(tids)} trees   {(time.time()-t0)/60:5.1f} min")

H = pd.concat([df.assign(split=t) for t, df in GEN.items()], ignore_index=True)
S = H.groupby("split", sort=False).agg(
    on_tree=("on_tree", "mean"), reach=("robot_reach", "mean"),
    attempts=("attempts", "mean"), premium=("premium", "mean"),
    utility=("utility", "mean"), seconds=("seconds", "mean")).round(2)
S["premium_per_reachable"] = (S.premium/S.reach).round(3)
print(f"\n10 seeds each\n")
print(S.to_string())

a, b = S.loc["training trees"], S.loc["held-out trees"]
print(f"\n  premium per reachable fruit: {a.premium_per_reachable:.3f} on trees it saw, "
      f"{b.premium_per_reachable:.3f} on trees it did not")
print(f"  the difference is {abs(a.premium_per_reachable - b.premium_per_reachable):.3f}, "
      f"against a per-tree spread of {H.groupby('split').premium.std().mean():.1f} fruit")
print("\n  Normalising by reachable fruit matters: the two sets of trees do not carry the same")
print("  crop, and comparing raw totals would report that difference as a generalisation gap.")


  training trees   20 trees     1.2 min
  held-out trees   30 trees     3.2 min

10 seeds each

                on_tree  reach  attempts  premium  utility  seconds  premium_per_reachable
split                                                                                     
training trees    120.0  62.15     61.72    48.90    44.75  1220.42                  0.787
held-out trees    120.0  62.57     62.08    49.35    45.23  1218.68                  0.789

  premium per reachable fruit: 0.787 on trees it saw, 0.789 on trees it did not
  the difference is 0.002, against a per-tree spread of 4.9 fruit

  Normalising by reachable fruit matters: the two sets of trees do not carry the same
  crop, and comparing raw totals would report that difference as a generalisation gap.


## 5. Freezing the run

Everything above depends on four files and three constants. They are written out with the results
so a figure can be traced back to the configuration that produced it — the alternative is what
happened earlier in this project, when three notebooks each carried their own copy of the
planning logic and their numbers stopped agreeing without anyone noticing.


In [7]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

a = axes[0]
for tag, df in PICKERS.items():
    a.scatter(df.seconds/60, df.premium, s=8, alpha=0.35, label=tag)
a.set_xlabel("minutes on the tree"); a.set_ylabel("premium fruit")
a.set_title("three ways to choose the next fruit")
a.legend(frameon=False, fontsize=8)

b = axes[1]
d = (A.premium - B.premium)
b.hist(d, bins=np.arange(d.min()-0.5, d.max()+1.5, 1), color="#2f6f4e")
b.axvline(0, color="#c0392b", lw=1)
b.set_xlabel("planner minus coverage, premium fruit")
b.set_ylabel("tree-seed pairs")
b.set_title(f"paired difference, mean {d.mean():+.2f}")
fig.tight_layout()
fig.savefig(OUT/"evaluation.png")
plt.close(fig)

spec = dict(
    configuration=dict(stops=PL.K_STATIONS, sweep=PL.SWEEP, threshold=PL.THRESHOLD,
                       arm=[PL.HALF_X, PL.LIFT], chooser="planner", picker="rule"),
    files=dict(planner=info["planner"], policy=info["policy"],
               outcome="outcome.pkl", dynamics="dynamics.joblib",
               module="src/planner.py"),
    data=dict(trees="trees_measured_pose.csv", picks="picks.csv",
              trees_train=[TREES_TRAIN[0], TREES_TRAIN[-1]],
              trees_eval=[TREES_EVAL[0], TREES_EVAL[-1]], seeds=N_SEEDS),
    layer1=dict(held_out_accuracy=round(float(acc), 2), rows=int(len(TE)),
                fruit=int(TE.apple_id.nunique()),
                worst_decile_gap=float(dec.gap.abs().max())),
    system={t: {k: round(float(v), 3) for k, v in
                dict(premium=df.premium.mean(), knocked=df.knocked.mean(),
                     utility=df.utility.mean(), seconds=df.seconds.mean()).items()}
            for t, df in PICKERS.items()},
    planner_vs_coverage={k: round(float(v), 4) for k, v in
                         zip(("difference", "lo", "hi", "p", "d"),
                             paired(A.premium, B.premium))})
(OUT/"config.json").write_text(json.dumps(spec, indent=1))
print(json.dumps(spec, indent=1))
print(f"\nwritten to {OUT}\n")
for f in sorted(OUT.iterdir()):
    print(f"  {f.name:<26} {f.stat().st_size/1024:8.1f} KB")



{
 "configuration": {
  "stops": 20,
  "sweep": true,
  "threshold": 0.0,
  "arm": [
   0.2,
   0.6
  ],
  "chooser": "planner",
  "picker": "rule"
 },
 "files": {
  "planner": "station_planner_k20.pt",
  "policy": "pick_policy.pt",
  "outcome": "outcome.pkl",
  "dynamics": "dynamics.joblib",
  "module": "src/planner.py"
 },
 "data": {
  "trees": "trees_measured_pose.csv",
  "picks": "picks.csv",
  "trees_train": [
   0,
   19
  ],
  "trees_eval": [
   20,
   49
  ],
  "seeds": 30
 },
 "layer1": {
  "held_out_accuracy": 94.07,
  "rows": 7052,
  "fruit": 1763,
  "worst_decile_gap": 0.026
 },
 "system": {
  "rate rule": {
   "premium": 49.607,
   "knocked": 3.28,
   "utility": 45.575,
   "seconds": 1218.592
  },
  "pick policy": {
   "premium": 33.353,
   "knocked": 0.118,
   "utility": 33.172,
   "seconds": 524.744
  },
  "threshold 0.9": {
   "premium": 37.033,
   "knocked": 0.093,
   "utility": 36.876,
   "seconds": 734.147
  }
 },
 "planner_vs_coverage": {
  "difference": 0.0,
  "lo"

### What this section supports and what it does not

The component figure is an accuracy and behaves like one. The system figures are draws from the
outcome model — they say what the chain does *if the model is right*, and the only check on that
premise is `04_fidelity`, which re-runs the same plans against the physics and reports the gap.

No comparison against human picking appears here, and none should be read into the per-hour
column: it is machine time on a machine that runs unattended.
